# Fine-tuning LoRA pour Stable Diffusion

In [4]:
import os
import gc
import torch
import torch.nn.functional as F
from torchvision import transforms
from diffusers import UNet2DConditionModel
from diffusers.loaders import AttnProcsLayers
from diffusers.models.attention_processor import LoRAAttnProcessor
from transformers import CLIPTextModel, CLIPTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

gc.collect()
torch.cuda.empty_cache()

## Configuration

In [5]:
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"
dataset_dir = "./Iwaria_Press_Dataset/images"
output_dir = "./output"
resolution = 512
train_batch_size = 2
num_train_epochs = 10
learning_rate = 1e-4
adam_weight_decay = 1e-2

os.makedirs(output_dir, exist_ok=True)

## 1. Chargement des composants du modèle de base

In [7]:
# tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")
# text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path, subfolder="text_encoder", torch_dtype=torch.float16)
# unet = UNet2DConditionModel.from_pretrained(pretrained_model_name_or_path, subfolder="unet", torch_dtype=torch.float16, low_cpu_mem_usage=True)

# # Geler les paramètres existants
# text_encoder.requires_grad_(False)
# unet.requires_grad_(False)

import gc, torch, os
from diffusers import UNet2DConditionModel
from diffusers.loaders import AttnProcsLayers
from diffusers.models.attention_processor import LoRAAttnProcessor
from transformers import CLIPTextModel, CLIPTokenizer

gc.collect()

pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"

tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path, subfolder="tokenizer")

text_encoder = CLIPTextModel.from_pretrained(
    pretrained_model_name_or_path, subfolder="text_encoder",
    torch_dtype=torch.float16, low_cpu_mem_usage=True
)
text_encoder.requires_grad_(False)
gc.collect()

unet = UNet2DConditionModel.from_pretrained(
    pretrained_model_name_or_path, subfolder="unet",
    torch_dtype=torch.float16, low_cpu_mem_usage=True,
    device_map="auto"   # répartit automatiquement entre RAM et GPU
)
unet.requires_grad_(False)
gc.collect()


Loading weights: 100%|██████████| 196/196 [00:01<00:00, 165.43it/s]
CLIPTextModel LOAD REPORT from: runwayml/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MemoryError: 

## 2. Injection des couches LoRA dans l'UNet

In [ ]:
lora_attn_procs = {}
for name, attn_processor in unet.attn_processors.items():
    cross_attention_dim = None if name.endswith("attn1.processor") else unet.config.cross_attention_dim
    if name.startswith("mid_block"):
        hidden_size = unet.config.block_out_channels[-1]
    elif name.startswith("up_blocks"):
        block_id = int(name.split(".")[1])
        hidden_size = list(reversed(unet.config.block_out_channels))[block_id]
    elif name.startswith("down_blocks"):
        block_id = int(name.split(".")[1])
        hidden_size = unet.config.block_out_channels[block_id]

    lora_attn_procs[name] = LoRAAttnProcessor(
        hidden_size=hidden_size, cross_attention_dim=cross_attention_dim, rank=4
    )

unet.set_attn_processor(lora_attn_procs)
lora_layers = AttnProcsLayers(unet.attn_processors)
unet.train()

## 3. Optimiseur

In [ ]:
optimizer = torch.optim.AdamW(
    lora_layers.parameters(),
    lr=learning_rate,
    weight_decay=adam_weight_decay,
)

## 4. Préparation du jeu de données

In [ ]:
dataset = load_dataset("imagefolder", data_dir=dataset_dir, split="train")

train_transforms = transforms.Compose([
    transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(resolution),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def preprocess(examples):
    images = [train_transforms(image.convert("RGB")) for image in examples["image"]]
    inputs = tokenizer(examples["text"], max_length=tokenizer.model_max_length, padding="max_length", truncation=True, return_tensors="pt")
    return {"pixel_values": images, "input_ids": inputs.input_ids}

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    input_ids = torch.stack([example["input_ids"] for example in examples])
    return {"pixel_values": pixel_values, "input_ids": input_ids}

train_dataset = dataset.with_transform(preprocess)
train_dataloader = torch.utils.data.DataLoader(
    train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=collate_fn
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entraînement sur : {device}")
unet.to(device)
text_encoder.to(device)

## 5. Boucle d'entraînement

In [ ]:
for epoch in range(num_train_epochs):
    progress_bar = tqdm(total=len(train_dataloader), desc=f"Epoch {epoch+1}")
    for step, batch in enumerate(train_dataloader):
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)

        noise = torch.randn_like(pixel_values)
        timesteps = torch.randint(0, 1000, (pixel_values.shape[0],), device=device).long()
        noisy_images = pixel_values + noise

        encoder_hidden_states = text_encoder(input_ids)[0]
        noise_pred = unet(noisy_images, timesteps, encoder_hidden_states).sample

        loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        progress_bar.update(1)
        progress_bar.set_postfix({"loss": loss.item()})

## 6. Sauvegarde des poids LoRA

In [ ]:
print(f"Sauvegarde du modèle dans {output_dir}...")
unet.save_attn_procs(output_dir)
print("Entraînement terminé avec succès !")